# 10. Results dashboard

This notebook turns the framework's results into clear figures for the report and for a non-technical audience: a pass/flag donut, the before-and-after drift-fix comparison, the per-batch quality stream, the Module 1 detector comparison, the ablation, and a plain-language summary table.

The full-scale figures read `results/fullscale_rolling.csv` and `fullscale_fixed.csv`. The module and ablation figures use the verified results recorded in PROJECT_CONTEXT (sections 4f and 4g). All figures are saved to `results/` at 150 dpi.

In [ ]:
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm

RES = '../results'
plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'medium',
    'axes.labelsize': 11, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.axisbelow': True, 'axes.edgecolor': '#555555',
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.dpi': 150,
})
# muted, publication-standard palette (seaborn 'deep' family) — not saturated/AI-looking
BLUE, ORANGE, GREEN, RED, GREY = '#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8C8C8C'
AMBER = ORANGE  # alias so existing references stay valid

# --- full-scale run (real data) ---
def _load(name, fail_fallback, n=61):
    p = f'{RES}/{name}'
    if os.path.exists(p):
        return pd.read_csv(p)
    # fallback so the notebook still renders if the CSV isn't present
    return pd.DataFrame({'batch': range(n), 'drift': np.nan, 'combined': np.nan,
                         'decision': ['FAIL']*fail_fallback + ['PASS']*(n-fail_fallback)})
roll  = _load('fullscale_rolling.csv', 3)
fixed = _load('fullscale_fixed.csv', 12)
N = len(roll)
roll_fail  = int((roll['decision']  == 'FAIL').sum())
fixed_fail = int((fixed['decision'] == 'FAIL').sum())

# --- verified experiment results (source: PROJECT_CONTEXT 4f/4g, notebook 08) ---
M1_CC = {  # detector: (ROC-AUC, PR-AUC) vs real Credit Card fraud
    'Isolation\nForest': (0.953, 0.132), 'LOF': (0.956, 0.356),
    'Z-score': (0.705, 0.004), 'Autoencoder': (0.956, 0.521)}
ABLATION = {  # variant: F1 (max-gate decision rule)
    'Full\nframework': 1.00, 'Remove\nanomaly': 1.00, 'Remove\ndrift': 0.80,
    'Remove\nmissing': 0.80, 'Equal\navg': 0.50, 'Severity\navg': 0.59,
    'Adaptive\navg': 0.57, 'Max\ngate': 1.00}
M3_MAR = 0.722
print('loaded: %d batches | rolling FAIL=%d | fixed FAIL=%d' % (N, roll_fail, fixed_fail))

## 1. The gate's decision on the whole data stream

In [ ]:
passed = N - roll_fail
fig, ax = plt.subplots(figsize=(6, 5))
wedges, _ = ax.pie([passed, roll_fail], colors=[GREEN, RED], startangle=90,
                   counterclock=False, wedgeprops=dict(width=0.42, edgecolor='white', linewidth=2))
ax.text(0, 0.12, f'{passed}', ha='center', fontsize=32, fontweight='medium', color='#333333')
ax.text(0, -0.16, f'of {N} batches passed', ha='center', fontsize=12, color='#555555')
ax.text(0, -0.34, f'{round(100*passed/N)}%', ha='center', fontsize=13, color=GREEN, fontweight='medium')
ax.legend(wedges, [f'Passed to QA — {passed} ({round(100*passed/N)}%)',
                   f'Flagged for review — {roll_fail} ({round(100*roll_fail/N)}%)'],
          loc='lower center', bbox_to_anchor=(0.5, -0.12), frameon=False, fontsize=11)
ax.set_title('Automated quality gate — outcome on 3,000,485 transactions', pad=16)
plt.tight_layout(); plt.savefig(f'{RES}/dash_gate_donut.png', bbox_inches='tight'); plt.show()

## 2. Before and after the drift fix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={'width_ratios': [1, 1.4]})
# (a) flagged-rate comparison
labels = ['Naive\n(fixed reference)', 'Fixed method\n(rolling reference)']
rates = [100*fixed_fail/N, 100*roll_fail/N]
bars = axes[0].bar(labels, rates, color=[GREY, BLUE], width=0.6)
for b, r, f in zip(bars, rates, [fixed_fail, roll_fail]):
    axes[0].text(b.get_x()+b.get_width()/2, r+1.5, f'{round(r)}%\n({f}/{N})',
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_ylabel('% of batches flagged'); axes[0].set_ylim(0, 110)
axes[0].set_title('Fewer false alarms after the fix')
# (b) per-batch quality score stream
axes[1].plot(fixed['batch'], fixed['combined'], color=GREY, lw=1.4, marker='.', ms=4, label='naive (fixed)')
axes[1].plot(roll['batch'],  roll['combined'],  color=BLUE, lw=1.6, marker='.', ms=4, label='fixed method (rolling)')
axes[1].axhline(0.5, color=AMBER, ls='--', lw=1.3, label='flag threshold')
axes[1].set_xlabel('incoming batch (50k rows each)'); axes[1].set_ylabel('quality score')
axes[1].set_title('Quality score across the whole stream'); axes[1].legend(frameon=False, fontsize=10)
for a in axes: a.grid(axis='y', ls=':', lw=0.7, color='#dddddd')
plt.tight_layout(); plt.savefig(f'{RES}/dash_before_after.png', bbox_inches='tight'); plt.show()

## 3. Module 1: how each anomaly detector performed on real fraud (Credit Card)

In [ ]:
names = list(M1_CC.keys()); roc = [v[0] for v in M1_CC.values()]; pr = [v[1] for v in M1_CC.values()]
x = np.arange(len(names)); w = 0.38
fig, ax = plt.subplots(figsize=(9, 5.6))
b1 = ax.bar(x-w/2, roc, w, color=BLUE, label='ROC-AUC (overall accuracy)')
b2 = ax.bar(x+w/2, pr,  w, color=AMBER, label='PR-AUC (accuracy on rare fraud)')
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f'{b.get_height():.2f}',
                ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylabel('score (higher = better)')
ax.set_ylim(0, 1.32)   # headroom so the legend clears the value labels
ax.set_title('Anomaly detectors vs genuine fraud labels')
ax.legend(loc='upper center', ncol=2, frameon=False, fontsize=9)
ax.grid(axis='y', ls=':', lw=0.7, color='#dddddd')
plt.tight_layout(); plt.savefig(f'{RES}/dash_module1_detectors.png', bbox_inches='tight'); plt.show()

## 4. Ablation: what each module and each combining rule contributes

In [ ]:
labels = list(ABLATION.keys()); vals = list(ABLATION.values())
# colour: averaging rules (weak) in grey, gate/module variants in blue, drops in amber
colours = [BLUE, BLUE, AMBER, AMBER, GREY, GREY, GREY, BLUE]
fig, ax = plt.subplots(figsize=(11, 5.6))
bars = ax.bar(labels, vals, color=colours, width=0.7)
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.02, f'{v:.2f}', ha='center', fontsize=9, fontweight='bold')
ax.set_ylim(0, 1.35)   # headroom so the legend sits above the bars
ax.set_ylabel('F1 (fault-detection quality)')
ax.set_title('Ablation: averaging dilutes a fault; the max-gate and full framework are best')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=BLUE, label='strong (gate / full)'),
                   Patch(color=AMBER, label='module removed'),
                   Patch(color=GREY, label='averaging rules (weak)')],
          frameon=False, fontsize=10, loc='upper center', ncol=3)
ax.grid(axis='y', ls=':', lw=0.7, color='#dddddd')
plt.tight_layout(); plt.savefig(f'{RES}/dash_ablation.png', bbox_inches='tight'); plt.show()

## 5. Plain-language summary table

In [ ]:
rows = [
    ['Unusual values\n(anomaly)', 'Amounts far outside the normal range (fraud/errors)',
     'Caught real fraud ~95% (ROC-AUC 0.95)'],
    ['Pattern change\n(drift)', "When new data stops looking like recent data",
     f'Flags only material change: {roll_fail}/{N} batches (5%)'],
    ['Missing info\n(missing values)', 'Predicts which records will have blank fields',
     f'Right ~7 in 10 (ROC-AUC {M3_MAR:.2f})'],
    ['The gate\n(all combined)', 'One PASS/FAIL decision per batch of data',
     f'Passed {N-roll_fail}/{N} batches, flagged {roll_fail} for a person'],
]
fig, ax = plt.subplots(figsize=(12, 3.2)); ax.axis('off')
tbl = ax.table(cellText=rows, colLabels=['Check', 'What it looks for (plain terms)', 'How well it worked'],
               cellLoc='left', colWidths=[0.20, 0.44, 0.36], loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10.5); tbl.scale(1, 2.4)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor('#c9c9c9'); cell.set_linewidth(0.6)
    if r == 0:
        cell.set_facecolor('#3b4a5a'); cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#f4f5f7')
ax.set_title('What the framework checks — and how it did', fontweight='bold', pad=12)
plt.tight_layout(); plt.savefig(f'{RES}/dash_summary_table.png', bbox_inches='tight'); plt.show()

## Reading the dashboard

Section 1 is the headline for a non-technical reader: the gate passed 95% of the data automatically and asked a person to look at only 5%. Section 2 shows the engineering value, where the naive gate flagged far more and the rolling-reference fix cut false alarms sharply while still catching genuine change. Sections 3 and 4 are the technical evidence, covering the detector comparison and the ablation that shows which module and which combining rule actually matter. Section 5 is a one-glance plain-language summary for slides. All six figures are saved in `results/` and can be dropped straight into the report or presentation.